# Create_Submissions_1

In [1]:
import os


# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

## Setup

In [64]:
import pandas as pd
import re
from pathlib import Path

DEFAULT_COMPETITION_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\kaggle"))
)
SUBMISSION_DATA_PATH = Path(
    os.path.abspath(os.path.join(os.getcwd(), r"data\submissions"))
)


# Define function to determine gender based on TeamID1
def determine_gender(team_id):
    if str(team_id).startswith("1"):  # Men's teams start with 1
        return "Men"
    elif str(team_id).startswith("3"):  # Women's teams start with 3
        return "Women"
    else:
        return None  # Handle unexpected cases


def extract_game_info(id_str: str) -> tuple[int, int, int]:
    """
    Extract season and team IDs from a Kaggle competition game ID string.

    Args:
        id_str (str): The game ID string formatted as "YYYY_Team1_Team2".

    Returns:
        tuple[int, int, int]: A tuple containing (Season, TeamID1, TeamID2).
    """
    try:
        year, team1, team2 = map(int, id_str.split("_"))
        return year, team1, team2
    except ValueError:
        raise ValueError(f"Unexpected ID format: {id_str}")


### Example usage:
# # Load the sample submission file
# submission_df = pd.read_csv('/mnt/data/SampleSubmissionStage1.csv')

# # Extract game info into new columns
# submission_df[['Season', 'TeamID1', 'TeamID2']] = submission_df['ID'].apply(extract_game_info).apply(pd.Series)


def extract_seed_value(seed_str: str) -> int:
    """
    Extracts the numeric seed value from an NCAA tournament seed string.

    Args:
        seed_str (str): The seed string (e.g., 'W01', 'Y12b').

    Returns:
        int: The extracted seed value, or 16 if extraction fails.
    """
    try:
        # Extract numeric part using regex
        match = re.search(r"\d+", seed_str)
        if match:
            return int(match.group())
        else:
            return 16  # Default seed value for unselected teams/errors
    except (ValueError, TypeError):
        return 16  # Default for unexpected cases


### Example usage:
# # Load the tournament seed data
# w_seed = pd.read_csv('/mnt/data/WNCAATourneySeeds.csv')
# m_seed = pd.read_csv('/mnt/data/MNCAATourneySeeds.csv')

# # Concatenate men's and women's data
# seed_df = pd.concat([m_seed, w_seed], axis=0, ignore_index=True)

# # Apply the function to extract seed values
# seed_df['SeedValue'] = seed_df['Seed'].apply(extract_seed_value)


# Function to merge results_df with seeds_df and tourney_round_lookup
def process_results(results_df, seeds_df, tourney_round_lookup):
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    results_df["WSeed"] = results_df["Seed"].str.rstrip("ab")
    results_df["LSeed"] = results_df["Seed_T2"].str.rstrip("ab")
    results_df["WSeedValue"] = results_df["SeedValue"]
    results_df["LSeedValue"] = results_df["SeedValue_T2"]

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)
    results_df["StrongSeedValue"] = results_df[["WSeedValue", "LSeedValue"]].min(axis=1)
    results_df["WeakSeedValue"] = results_df[["WSeedValue", "LSeedValue"]].max(axis=1)

    # Create SeedMatchup column using vectorized string operations
    results_df["SeedMatchup"] = (
        "No. "
        + results_df["StrongSeedValue"].astype(str)
        + " vs. No. "
        + results_df["WeakSeedValue"].astype(str)
    )

    results_df = results_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    results_df = results_df[
        [
            "Season",
            "DayNum",
            "WTeamID",
            "WSeed",
            "WScore",
            "LTeamID",
            "LSeed",
            "LScore",
            "WLoc",
            "NumOT",
            "Round",
            "Slot",
            "SeedMatchup",
            "WSeedValue",
            "LSeedValue",
            "StrongSeedValue",
            "WeakSeedValue",
        ]
    ]

    return results_df


### Example usage:
# # Load data
# m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
# w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
# m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
# w_results = pd.read_csv(r"data\kaggle\WNCAATourneyCompactResults.csv")
# tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

# # Extract numeric seed values
# m_seed['SeedValue'] = m_seed['Seed'].apply(extract_seed_value)
# w_seed['SeedValue'] = w_seed['Seed'].apply(extract_seed_value)

# # Merge results_df with seeds_df and tourney_round_lookup
# m_results = process_results(m_results, m_seed, tourney_round_lookup)
# w_results = process_results(w_results, w_seed, tourney_round_lookup)


# Function to merge sub_df with seeds_df and tourney_round_lookup
def process_submission(sub_df, seeds_df, tourney_round_lookup):
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID1"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    sub_df = sub_df.merge(
        seeds_df,
        left_on=["Season", "TeamID2"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    sub_df["Seed1"] = sub_df["Seed"].str.rstrip("ab").fillna("")
    sub_df["Seed2"] = sub_df["Seed_T2"].str.rstrip("ab").fillna("")
    sub_df["SeedValue1"] = sub_df["SeedValue"].astype("Int64")
    sub_df["SeedValue2"] = sub_df["SeedValue_T2"].astype("Int64")

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    sub_df["StrongSeed"] = sub_df[["Seed1", "Seed2"]].min(axis=1)
    sub_df["WeakSeed"] = sub_df[["Seed1", "Seed2"]].max(axis=1)
    sub_df["StrongSeedValue"] = sub_df[["SeedValue1", "SeedValue2"]].min(axis=1)
    sub_df["WeakSeedValue"] = sub_df[["SeedValue1", "SeedValue2"]].max(axis=1)

    # Create SeedMatchup column using vectorized string operations
    sub_df["SeedMatchup"] = (
        "No. "
        + sub_df["StrongSeedValue"].astype(str)
        + " vs. No. "
        + sub_df["WeakSeedValue"].astype(str)
    )

    sub_df = sub_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    sub_df = sub_df[
        [
            "ID",
            "Pred",
            "Season",
            "TeamID1",
            "Seed1",
            "TeamID2",
            "Seed2",
            "Round",
            "Slot",
            "SeedMatchup",
            "SeedValue1",
            "SeedValue2",
            "StrongSeedValue",
            "WeakSeedValue",
        ]
    ]

    return sub_df


### Example usage:
# # Load data
# m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
# w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
# submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")
# tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

# # Extract numeric seed values
# m_seed['SeedValue'] = m_seed['Seed'].apply(extract_seed_value)
# w_seed['SeedValue'] = w_seed['Seed'].apply(extract_seed_value)

# # Extract game info
# submission_df[['Season', 'TeamID1', 'TeamID2']] = submission_df['ID'].apply(extract_game_info).apply(pd.Series)

# # Assign Gender column
# submission_df['Gender'] = submission_df['TeamID1'].apply(determine_gender)

# # Split into men's and women's submission dataframes
# m_submission = submission_df[submission_df['Gender'] == 'Men'].copy()
# w_submission = submission_df[submission_df['Gender'] == 'Women'].copy()

# # Merge sub_df with seeds_df and tourney_round_lookup
# m_submission = process_submission(m_submission, m_seed, tourney_round_lookup)
# w_submission = process_submission(w_submission, w_seed, tourney_round_lookup)

## `Round1_SeedProbabilities.csv`

Start with kaggle\SampleSubmissionStage1.csv; Update both men's and women's Round 1 matches with historical seed probabilities.

In [44]:
# Import Data
m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
w_seed = pd.read_csv(r"data\kaggle\WNCAATourneySeeds.csv")
m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
w_results = pd.read_csv(r"data\kaggle\WNCAATourneyCompactResults.csv")
tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

### Determine Round 1 Seed Probabilities

`m_round1_percents` and `w_round1_percents`

In [45]:
# Extract numeric seed values
m_seed["SeedValue"] = m_seed["Seed"].apply(extract_seed_value)
w_seed["SeedValue"] = w_seed["Seed"].apply(extract_seed_value)

# Merge results_df with seeds_df and tourney_round_lookup
m_results = process_results(m_results, m_seed, tourney_round_lookup)
w_results = process_results(w_results, w_seed, tourney_round_lookup)

In [46]:
def get_round1_percents(results_df):
    # Filter results_df for Round 1 games
    round_df = results_df[results_df["Round"] == 1].copy()

    # Determine Wins and Losses
    round_df["Win"] = (round_df["WSeedValue"] == round_df["StrongSeedValue"]).astype(
        int
    )
    round_df["Loss"] = (round_df["WSeedValue"] == round_df["WeakSeedValue"]).astype(int)

    # Group by SeedMatchup and calculate Wins, Losses, and WinningPercentage
    round1_percents_df = (
        round_df.groupby("SeedMatchup")
        .agg(Wins=("Win", "sum"), Losses=("Loss", "sum"))
        .reset_index()
    )

    # Calculate WinningPercentage
    round1_percents_df["WinningPercentage"] = round1_percents_df["Wins"] / (
        round1_percents_df["Wins"] + round1_percents_df["Losses"]
    )

    return round1_percents_df

In [47]:
m_round1_percents = get_round1_percents(m_results)
w_round1_percents = get_round1_percents(w_results)

In [48]:
m_round1_percents

,SeedMatchup,Wins,Losses,WinningPercentage
0,No. 1 vs. No. 16,154,2,0.987179
1,No. 2 vs. No. 15,145,11,0.929487
2,No. 3 vs. No. 14,133,23,0.852564
3,No. 4 vs. No. 13,123,33,0.788462
4,No. 5 vs. No. 12,101,55,0.647436
5,No. 6 vs. No. 11,95,61,0.608974
6,No. 7 vs. No. 10,95,60,0.612903
7,No. 8 vs. No. 9,75,81,0.480769


In [49]:
w_round1_percents

,SeedMatchup,Wins,Losses,WinningPercentage
0,No. 1 vs. No. 16,103,1,0.990385
1,No. 2 vs. No. 15,104,0,1.000000
2,No. 3 vs. No. 14,104,0,1.000000
3,No. 4 vs. No. 13,98,6,0.942308
4,No. 5 vs. No. 12,82,22,0.788462
5,No. 6 vs. No. 11,69,35,0.663462
6,No. 7 vs. No. 10,69,35,0.663462
7,No. 8 vs. No. 9,54,50,0.519231


### Update both men's and women's Round 1 matches with historical seed probabilities.

In [58]:
# Load data
submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")

# Extract game info
submission_df[["Season", "TeamID1", "TeamID2"]] = (
    submission_df["ID"].apply(extract_game_info).apply(pd.Series)
)

# Assign Gender column
submission_df["Gender"] = submission_df["TeamID1"].apply(determine_gender)

# Split into men's and women's submission dataframes
m_submission = submission_df[submission_df["Gender"] == "Men"].copy()
w_submission = submission_df[submission_df["Gender"] == "Women"].copy()

# Merge sub_df with seeds_df and tourney_round_lookup
m_submission = process_submission(m_submission, m_seed, tourney_round_lookup)
w_submission = process_submission(w_submission, w_seed, tourney_round_lookup)

In [60]:
m_submission.head()

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed2,Round,Slot,SeedMatchup,SeedValue1,SeedValue2,StrongSeedValue,WeakSeedValue
0,2021_1101_1102,0.5,2021,1101,W14,1102,,NaN,NaN,No. 14 vs. No. 14,14,<NA>,14,14
1,2021_1101_1103,0.5,2021,1101,W14,1103,,NaN,NaN,No. 14 vs. No. 14,14,<NA>,14,14
2,2021_1101_1104,0.5,2021,1101,W14,1104,W02,3.0,R3W2,No. 2 vs. No. 14,14,2,2,14
3,2021_1101_1105,0.5,2021,1101,W14,1105,,NaN,NaN,No. 14 vs. No. 14,14,<NA>,14,14
4,2021_1101_1106,0.5,2021,1101,W14,1106,,NaN,NaN,No. 14 vs. No. 14,14,<NA>,14,14


In [61]:
def update_pred(sub_df, round1_percents_df):
    """
    Updates the 'Pred' column in sub_df with 'WinningPercentage' from round1_percents_df
    for rows where 'Round' == 1.

    Args:
        sub_df (pd.DataFrame): DataFrame containing 'SeedMatchup', 'Round', and 'Pred'.
        round1_percents_df (pd.DataFrame): DataFrame containing 'SeedMatchup' and 'WinningPercentage'.

    Returns:
        pd.DataFrame: Updated sub_df with 'Pred' values replaced where applicable.
    """
    # Merge sub_df with the relevant columns of round1_percents_df on 'SeedMatchup'
    merged_df = pd.merge(
        sub_df,
        round1_percents_df[["SeedMatchup", "WinningPercentage"]],
        on="SeedMatchup",
        how="left",
    )

    # Update 'Pred' based on SeedValue comparison for Round 1 matchups
    mask = merged_df["Round"] == 1
    merged_df.loc[
        mask & (merged_df["SeedValue1"] < merged_df["SeedValue2"]), "Pred"
    ] = merged_df.loc[
        mask & (merged_df["SeedValue1"] < merged_df["SeedValue2"]), "WinningPercentage"
    ]
    merged_df.loc[
        mask & (merged_df["SeedValue1"] > merged_df["SeedValue2"]), "Pred"
    ] = (
        1
        - merged_df.loc[
            mask & (merged_df["SeedValue1"] > merged_df["SeedValue2"]),
            "WinningPercentage",
        ]
    )

    # Drop the 'WinningPercentage' column as it's no longer needed
    sub_df_updated = merged_df.drop(columns=["WinningPercentage"])

    return sub_df_updated

In [62]:
m_submission = update_pred(m_submission, m_round1_percents)
w_submission = update_pred(w_submission, w_round1_percents)

In [63]:
m_submission[
    (m_submission["StrongSeedValue"] == 1) & (m_submission["Round"] == 1)
].head()

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed2,Round,Slot,SeedMatchup,SeedValue1,SeedValue2,StrongSeedValue,WeakSeedValue
3167,2021_1111_1211,0.012821,2021,1111,X16,1211,X01,1.0,R1X1,No. 1 vs. No. 16,16,1,1,16
6812,2021_1124_1216,0.987179,2021,1124,Z01,1216,Z16,1.0,R1Z1,No. 1 vs. No. 16,1,16,1,16
21295,2021_1180_1228,0.012821,2021,1180,Y16,1228,Y01,1.0,R1Y1,No. 1 vs. No. 16,16,1,1,16
29499,2021_1211_1313,0.987179,2021,1211,X01,1313,X16,1.0,R1X1,No. 1 vs. No. 16,1,16,1,16
42839,2021_1276_1291,0.987179,2021,1276,W01,1291,W16,1.0,R1W1,No. 1 vs. No. 16,1,16,1,16


In [65]:
Round1_SeedProbabilities = pd.concat(
    [m_submission[["ID", "Pred"]], w_submission[["ID", "Pred"]]],
    axis=0,
    ignore_index=True,
)

In [66]:
Round1_SeedProbabilities.to_csv(
    SUBMISSION_DATA_PATH / "Round1_SeedProbabilities.csv", index=False
)

## `Round1_BetExplorer.csv`

Start with kaggle\SampleSubmissionStage1.csv; Update men's Round 1 matches with BetExplorer odds/probabilities.

In [70]:
# Import Data
m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
# m_results = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
m_results = pd.read_csv(r"data\betexplorer\MNCAATourneyCompactResultsProbs.csv")
tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")

In [71]:
m_results.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,LOdds,match_date,WProbability,LProbability
0,1985,136,1116,63,1234,54,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
1,1985,136,1120,59,1345,58,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
2,1985,136,1207,68,1250,43,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
3,1985,136,1229,58,1425,55,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
4,1985,136,1242,49,1325,38,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN


In [72]:
# Function to merge results_df with seeds_df and tourney_round_lookup
def process_betexplorer_results(results_df, seeds_df, tourney_round_lookup):
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T1"),
    )
    results_df = results_df.merge(
        seeds_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="left",
        suffixes=("", "_T2"),
    )

    results_df["WSeed"] = results_df["Seed"].str.rstrip("ab")
    results_df["LSeed"] = results_df["Seed_T2"].str.rstrip("ab")
    results_df["WSeedValue"] = results_df["SeedValue"]
    results_df["LSeedValue"] = results_df["SeedValue_T2"]

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)
    results_df["StrongSeedValue"] = results_df[["WSeedValue", "LSeedValue"]].min(axis=1)
    results_df["WeakSeedValue"] = results_df[["WSeedValue", "LSeedValue"]].max(axis=1)

    # Create SeedMatchup column using vectorized string operations
    results_df["SeedMatchup"] = (
        "No. "
        + results_df["StrongSeedValue"].astype(str)
        + " vs. No. "
        + results_df["WeakSeedValue"].astype(str)
    )

    results_df = results_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    # Drop unnecessary columns and return
    results_df = results_df[
        [
            "Season",
            "DayNum",
            "WTeamID",
            "WSeed",
            "WScore",
            "LTeamID",
            "LSeed",
            "LScore",
            "WLoc",
            "NumOT",
            "Round",
            "Slot",
            "WProbability",
            "LProbability",
            "SeedMatchup",
            "WSeedValue",
            "LSeedValue",
            "StrongSeedValue",
            "WeakSeedValue",
        ]
    ]

    return results_df

In [73]:
# Extract numeric seed values
m_seed["SeedValue"] = m_seed["Seed"].apply(extract_seed_value)

# Merge results_df with seeds_df and tourney_round_lookup
m_results = process_betexplorer_results(m_results, m_seed, tourney_round_lookup)

In [95]:
m_results[
    (m_results["Round"] == 1)
    & (m_results["Season"] >= 2021)
    & (m_results["WSeedValue"] == 11)
]

,Season,DayNum,WTeamID,WSeed,WScore,LTeamID,LSeed,LScore,WLoc,NumOT,Round,Slot,WProbability,LProbability,SeedMatchup,WSeedValue,LSeedValue,StrongSeedValue,WeakSeedValue
2266,2021,137,1393,Y11,78,1361,Y06,62,N,0,1.0,R1Y6,0.404488,0.595512,No. 6 vs. No. 11,11,6,6,11
2284,2021,138,1417,W11,73,1140,W06,62,N,0,1.0,R1W6,0.395941,0.604059,No. 6 vs. No. 11,11,6,6,11
2327,2022,136,1276,Z11,75,1161,Z06,63,N,0,1.0,R1Z6,0.544295,0.455705,No. 6 vs. No. 11,11,6,6,11
2342,2022,137,1235,Y11,59,1261,Y06,54,N,0,1.0,R1Y6,0.374190,0.625810,No. 6 vs. No. 11,11,6,6,11
2345,2022,137,1323,X11,78,1104,X06,64,N,0,1.0,R1X6,0.362345,0.637655,No. 6 vs. No. 11,11,6,6,11
2416,2023,137,1338,Y11,59,1235,Y06,41,N,0,1.0,R1Y6,0.347790,0.652210,No. 6 vs. No. 11,11,6,6,11
2458,2024,136,1182,W11,71,1140,W06,67,N,0,1.0,R1W6,0.192392,0.807608,No. 6 vs. No. 11,11,6,6,11
2464,2024,136,1301,Z11,80,1403,Z06,67,N,0,1.0,R1Z6,0.351767,0.648233,No. 6 vs. No. 11,11,6,6,11
2467,2024,136,1332,Y11,87,1376,Y06,73,N,0,1.0,R1Y6,0.573188,0.426812,No. 6 vs. No. 11,11,6,6,11


### Update men's Round 1 matches with BetExplorer odds/probabilities.

In [77]:
# Load data
submission_df = pd.read_csv(r"data\kaggle\SampleSubmissionStage1.csv")

# Extract game info
submission_df[["Season", "TeamID1", "TeamID2"]] = (
    submission_df["ID"].apply(extract_game_info).apply(pd.Series)
)

# Merge sub_df with seeds_df and tourney_round_lookup
submission_df = process_submission(submission_df, m_seed, tourney_round_lookup)

In [80]:
submission_df[submission_df["Round"] == 1]

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed2,Round,Slot,SeedMatchup,SeedValue1,SeedValue2,StrongSeedValue,WeakSeedValue
278,2021_1101_1400,0.5,2021,1101,W14,1400,W03,1.0,R1W3,No. 3 vs. No. 14,14,3,3,14
1150,2021_1104_1233,0.5,2021,1104,W02,1233,W15,1.0,R1W2,No. 2 vs. No. 15,2,15,2,15
3167,2021_1111_1211,0.5,2021,1111,X16,1211,X01,1.0,R1X1,No. 1 vs. No. 16,16,1,1,16
4788,2021_1116_1159,0.5,2021,1116,Z03,1159,Z14,1.0,R1Z3,No. 3 vs. No. 14,3,14,3,14
6812,2021_1124_1216,0.5,2021,1124,Z01,1216,Z16,1.0,R1Z1,No. 1 vs. No. 16,1,16,1,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
432519,2024_1332_1376,0.5,2024,1332,Y11,1376,Y06,1.0,R1Y6,No. 6 vs. No. 11,11,6,6,11
436208,2024_1361_1412,0.5,2024,1361,W05,1412,W12,1.0,R1W5,No. 5 vs. No. 12,5,12,5,12
438754,2024_1389_1397,0.5,2024,1389,Y15,1397,Y02,1.0,R1Y2,No. 2 vs. No. 15,15,2,2,15
439281,2024_1395_1429,0.5,2024,1395,Y09,1429,Y08,1.0,R1Y8,No. 8 vs. No. 9,9,8,8,9


In [82]:
def update_pred_with_betexplorer(sub_df, results_df):
    """
    Updates the 'Pred' column in sub_df using betting probabilities from results_df.

    Args:
        sub_df (pd.DataFrame): DataFrame containing 'Season', 'TeamID1', 'TeamID2', and 'Pred'.
        results_df (pd.DataFrame): DataFrame containing 'Season', 'WTeamID', 'LTeamID', 'WProbability', 'LProbability', and 'Round'.

    Returns:
        pd.DataFrame: Updated sub_df with 'Pred' values replaced where applicable.
    """
    # Filter results_df to only include Round 1 matchups
    results_df = results_df[results_df["Round"] == 1].copy()

    # Create TeamID1 and TeamID2 ensuring the smaller ID is first
    results_df["TeamID1"] = results_df[["WTeamID", "LTeamID"]].min(axis=1)
    results_df["TeamID2"] = results_df[["WTeamID", "LTeamID"]].max(axis=1)

    # Assign Probability based on which team is TeamID1 using a vectorized approach
    results_df["Probability"] = results_df["WProbability"] * (
        results_df["TeamID1"] == results_df["WTeamID"]
    ) + results_df["LProbability"] * (results_df["TeamID1"] == results_df["LTeamID"])

    # Merge with sub_df on Season, TeamID1, and TeamID2
    merged_df = sub_df.merge(
        results_df[["Season", "TeamID1", "TeamID2", "Probability"]],
        on=["Season", "TeamID1", "TeamID2"],
        how="left",
    )

    # Update Pred based on Probability where available
    merged_df["Pred"] = merged_df["Probability"].combine_first(merged_df["Pred"])

    # Drop Probability column as it's no longer needed
    sub_df_updated = merged_df.drop(columns=["Probability"])

    return sub_df_updated

In [83]:
Round1_BetExplorer = update_pred_with_betexplorer(submission_df, m_results)

In [87]:
Round1_BetExplorer[
    (Round1_BetExplorer["StrongSeedValue"] == 2) & (Round1_BetExplorer["Round"] == 1)
]

,ID,Pred,Season,TeamID1,Seed1,TeamID2,Seed2,Round,Slot,SeedMatchup,SeedValue1,SeedValue2,StrongSeedValue,WeakSeedValue
1150,2021_1104_1233,0.944287,2021,1104,W02,1233,W15,1.0,R1W2,No. 2 vs. No. 15,2,15,2,15
15241,2021_1156_1222,0.025424,2021,1156,Y15,1222,Y02,1.0,R1Y2,No. 2 vs. No. 15,15,2,2,15
29914,2021_1213_1234,0.084546,2021,1213,X15,1234,X02,1.0,R1X2,No. 2 vs. No. 15,15,2,2,15
50581,2021_1326_1331,0.934877,2021,1326,Z02,1331,Z15,1.0,R1Z2,No. 2 vs. No. 15,2,15,2,15
125418,2022_1120_1240,0.942174,2022,1120,Y02,1240,Y15,1.0,R1Y2,No. 2 vs. No. 15,2,15,2,15
139330,2022_1168_1181,0.035538,2022,1168,X15,1181,X02,1.0,R1X2,No. 2 vs. No. 15,15,2,2,15
141335,2022_1174_1437,0.066636,2022,1174,Z15,1437,Z02,1.0,R1Z2,No. 2 vs. No. 15,15,2,2,15
159322,2022_1246_1389,0.963535,2022,1246,W02,1389,W15,1.0,R1W2,No. 2 vs. No. 15,2,15,2,15
250260,2023_1112_1343,0.922490,2023,1112,X02,1343,X15,1.0,R1X2,No. 2 vs. No. 15,2,15,2,15
264197,2023_1159_1400,0.104298,2023,1159,Y15,1400,Y02,1.0,R1Y2,No. 2 vs. No. 15,15,2,2,15


In [86]:
Round1_BetExplorer[["ID", "Pred"]].to_csv(
    SUBMISSION_DATA_PATH / "Round1_BetExplorer.csv", index=False
)